# Waveform quality control and preprocessing

Applies component-completeness checks and shared conditioning to raw event-station windows. Outputs QC manifests and accepted windows for pseudo-labelling.


In [5]:
# ==========================================
# 1. IMPORTS AND PATHS
# ==========================================
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def _find_repo_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "data" / "raw").is_dir() and (p / "src").is_dir():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd().resolve())

# preprocessing_utils is the SINGLE SOURCE of the conditioning (identical
# preprocessing in labelling and in fine-tuning).
sys.path.insert(0, str(REPO_ROOT / "src" / "preprocessing"))
import preprocessing_utils as pp  # noqa: E402

CATALOG_DIR = REPO_ROOT / "data" / "raw" / "catalog"
WAVEFORMS_DIR = REPO_ROOT / "data" / "raw" / "waveforms"
INTERIM_DIR = REPO_ROOT / "data" / "interim" / "preprocessing"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root:   {REPO_ROOT}")
print(f"Waveforms:   {WAVEFORMS_DIR}")
print(f"Inventories: {INTERIM_DIR}")
print(f"Window: {pp.NPTS} samples @ {pp.SAMPLING_RATE_HZ:.0f} Hz "
      f"({pp.WINDOW_SECONDS} s), theoretical P at sample {pp.THEORETICAL_P_SAMPLE}")

Repo root:   M:\Github\TFM_SeismicPhasePicker_ColombianAndes
Waveforms:   M:\Github\TFM_SeismicPhasePicker_ColombianAndes\data\raw\waveforms
Inventories: M:\Github\TFM_SeismicPhasePicker_ColombianAndes\data\interim\preprocessing
Window: 15001 samples @ 100 Hz (150 s), theoretical P at sample 3000


In [6]:
# ==========================================
# 2. CONFIGURATION
# ==========================================
# Stations to process. By default the station folders present under
# data/raw/waveforms/ are auto-detected (single-level listing, cheap). To run
# only one or a few, set STATIONS_TO_PROCESS manually, e.g. ["ZAR"].
EXCLUDED = {"PTB1", "RIO2C", "YPLC"}  # stations not downloaded

STATIONS_TO_PROCESS = sorted(
    d.name for d in WAVEFORMS_DIR.iterdir()
    if d.is_dir() and d.name not in EXCLUDED
)

SAVE_PER_STATION = True   # write one CSV per station
KEEP_DATA = False         # do not hold the waveforms in RAM during QC

print(f"{len(STATIONS_TO_PROCESS)} stations to process:")
print(", ".join(STATIONS_TO_PROCESS))

22 stations to process:
BRJC, CBOC, CHI, DBB, GUY2C, JAMC, MAL, NOR, ORTC, PAL, PIZC, PRA, PTB, PTGC, QUBC, RUS, SPBC, TAM, URE, URMC, YOT, ZAR


In [7]:
# ==========================================
# 3. QC + PREPROCESSING PER STATION
# ==========================================
# For each station: group the SAC files by window (timestamp), run the
# completeness QC + preprocessing on every window and record the outcome.
# Only the station folder is listed (non-recursive glob) to avoid stalling
# the network drive.
import time


def process_station(station_code: str) -> pd.DataFrame:
    station_dir = WAVEFORMS_DIR / station_code
    windows = pp.group_station_windows(station_dir)  # {timestamp: {channel: path}}

    records = []
    for timestamp, files in windows.items():
        res = pp.qc_preprocess_window(
            files, station_code, timestamp, return_data=KEEP_DATA
        )
        records.append(res.as_record())

    df = pd.DataFrame(records)
    if SAVE_PER_STATION and not df.empty:
        df.to_csv(INTERIM_DIR / f"window_inventory_{station_code}.csv", index=False)
    return df


all_frames = []
for i, sta in enumerate(STATIONS_TO_PROCESS, 1):
    t0 = time.time()
    df_sta = process_station(sta)
    n_ok = int((df_sta["status"] == "ok").sum()) if not df_sta.empty else 0
    n_tot = len(df_sta)
    dt = time.time() - t0
    print(f"[{i:>2}/{len(STATIONS_TO_PROCESS)}] {sta:<6} "
          f"windows={n_tot:>6}  valid={n_ok:>6}  "
          f"discarded={n_tot - n_ok:>5}  ({dt:5.1f}s)")
    all_frames.append(df_sta)

inventory = pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()
inventory.to_csv(INTERIM_DIR / "window_inventory.csv", index=False)
print(f"\nCombined inventory: {len(inventory)} windows -> "
      f"{INTERIM_DIR / 'window_inventory.csv'}")

[ 1/22] BRJC   windows= 11901  valid= 11829  discarded=   72  (470.1s)
[ 2/22] CBOC   windows= 13647  valid= 13585  discarded=   62  (606.7s)
[ 3/22] CHI    windows=  9822  valid=  9767  discarded=   55  (397.6s)
[ 4/22] DBB    windows=  5817  valid=  5785  discarded=   32  (260.6s)
[ 5/22] GUY2C  windows= 13127  valid= 13067  discarded=   60  (526.9s)
[ 6/22] JAMC   windows=  5915  valid=  5886  discarded=   29  (251.3s)
[ 7/22] MAL    windows=  2986  valid=  2972  discarded=   14  (118.8s)
[ 8/22] NOR    windows= 15108  valid= 15007  discarded=  101  (610.1s)
[ 9/22] ORTC   windows=  9096  valid=  9031  discarded=   65  (406.2s)
[10/22] PAL    windows= 10766  valid= 10683  discarded=   83  (438.9s)
[11/22] PIZC   windows=  5956  valid=  5482  discarded=  474  (261.7s)
[12/22] PRA    windows=  9579  valid=  9554  discarded=   25  (404.7s)
[13/22] PTB    windows=  9140  valid=  9103  discarded=   37  (363.5s)
[14/22] PTGC   windows=  6768  valid=  6698  discarded=   70  (280.1s)
[15/22

In [8]:
# ==========================================
# 4. GLOBAL SUMMARY
# ==========================================
if not inventory.empty:
    n_total = len(inventory)
    n_ok = int((inventory["status"] == "ok").sum())
    print(f"Total windows:     {n_total}")
    print(f"Valid windows:     {n_ok}  ({100 * n_ok / n_total:.1f} %)")
    print(f"Discarded windows: {n_total - n_ok}  "
          f"({100 * (n_total - n_ok) / n_total:.1f} %)\n")

    print("Discard reason:")
    print(inventory.loc[inventory["status"] != "ok", "reason"]
          .value_counts().to_string())

    print("\nValid windows per station:")
    summary = (inventory.assign(ok=(inventory["status"] == "ok").astype(int))
               .groupby("station_code")
               .agg(windows=("trace_name", "size"), valid=("ok", "sum")))
    summary["discarded"] = summary["windows"] - summary["valid"]
    summary["rate_%"] = (100 * summary["valid"] / summary["windows"]).round(1)
    print(summary.sort_index().to_string())
else:
    print("Empty inventory: check STATIONS_TO_PROCESS and the paths.")

Total windows:     193410
Valid windows:     191488  (99.0 %)
Discarded windows: 1922  (1.0 %)

Discard reason:
reason
too_short                1476
missing_components        416
start_misaligned           27
read_error:SacIOError       3

Valid windows per station:
              windows  valid  discarded  rate_%
station_code                                   
BRJC            11901  11829         72    99.4
CBOC            13647  13585         62    99.5
CHI              9822   9767         55    99.4
DBB              5817   5785         32    99.4
GUY2C           13127  13067         60    99.5
JAMC             5915   5886         29    99.5
MAL              2986   2972         14    99.5
NOR             15108  15007        101    99.3
ORTC             9096   9031         65    99.3
PAL             10766  10683         83    99.2
PIZC             5956   5482        474    92.0
PRA              9579   9554         25    99.7
PTB              9140   9103         37    99.6
PTGC         